In [ ]:
# Install dependencies
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

import threading
import subprocess
import time
import re

# Select model and set environment variables
MODEL = "qwen3.5:9b"
%env OLLAMA_CONTEXT_LENGTH=128000
%env OLLAMA_HOST=0.0.0.0
%env OLLAMA_KEEP_ALIVE=-1

# Start ollama
def run_ollama_serve():
    subprocess.Popen(["ollama", "serve"])
thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

# Download model
!ollama pull {MODEL}
# Load model into RAM
subprocess.run(["ollama", "run", MODEL, "hello"], capture_output=True)

# Set up access via link through Cloudflare
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
cloudflared_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
url = None
for out in cloudflared_proc.stdout:
    match = re.search(r'(https://.*\.trycloudflare\.com)', out)
    if match:
        url = match.group(1)
        break
print(url)